In [ ]:
import scanpy as sc
import pandas as pd
from scipy.io import mmread

In [ ]:
# 1. Read the matrix
counts = mmread("counts.mtx").tocsr().T 

# 2. Load the index files
genes = pd.read_csv('genes.tsv', sep='\t', header=None)[0].values
cells = pd.read_csv('barcodes.tsv', sep='\t', header=None)[0].values
meta = pd.read_csv('metadata.tsv', sep='\t', index_col=0)

# FIX 1: Clean the meta index name
meta.index.name = "barcode"

# 3. Build AnnData 
# FIX 2: Create the index objects EXPLICITLY as strings/None
obs_index = pd.Index(cells, name="barcode")
var_index = pd.Index(genes, name="gene_symbols")


adata = sc.AnnData(
    X   = counts,
    obs = pd.DataFrame(index=obs_index),
    var = pd.DataFrame(index=var_index)
)

# 4. Attach metadata
# Since both indexes are now named "barcode", the join will be seamless
adata.obs = adata.obs.join(meta, how="left")

print("Columns in meta:", meta.columns.tolist())
print("First few rows of meta:\n", meta.head())

# 5. Factorize and finalize
adata.obs["Ground_Truth_Cluster"], uniques = pd.factorize(adata.obs["Ground_Truth_Celltype"])
adata.obs["Ground_Truth_Cluster"] += 1 

#Save raw counts before normalization (like RNA@counts in Seurat)
adata.raw = adata.copy()

print("Cluster mapping:", dict(enumerate(uniques, start=1)))

print(adata.obs['Ground_Truth_Celltype'].value_counts())

print(adata)


In [ ]:
# --- GLOBAL PREPROCESSING & RAW LAYERS ---
# Filter genes/cells first so the 'counts' layer matches the final object size
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
    
# Layer 1: Raw Integers (Needed for tools like scVI or CellAssign)
adata.layers["counts"] = adata.X.copy()
    
# Select HVGs using seurat_v3 (Requires raw counts)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3')
    
# Normalization
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
    
# Layer 2: Log-Normalized (Needed for AI Agents / Literature Search)
adata.layers["log1p"] = adata.X.copy()

   
   
# --- 3. FORK A: SUBSETTED (The Standard Pipeline) ---
print(f"  → Creating Subsetted version (HVGs only)...")
adata_sub = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata_sub, max_value=10)
sc.tl.pca(adata_sub, n_comps=50, mask_var=None) # PCA on 2000 genes
    
adata_sub.write("Cellbench-10x-5cl_anndata_subset.h5ad")
del adata_sub # Free memory immediately

    
    
# --- 4. FORK B: FULL (The Stress-Test Pipeline) ---
print(f"  → Creating Full version (All genes, Scaled + PCA)...")
# Using the original 'adata' variable which still contains all genes
sc.pp.scale(adata, max_value=10)
# PCA on ALL ~20,000 genes to test noise-tolerance
sc.tl.pca(adata, n_comps=50, mask_var=None) 
    
adata.write("Cellbench-10x-5cl_full.h5ad")
